In [1]:
!git clone https://github.com/lshek22/facial-expression-recognition.git
%cd facial-expression-recognition
!pip install wandb -q

Cloning into 'facial-expression-recognition'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 22 (delta 3), reused 13 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 57.50 KiB | 1.98 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/facial-expression-recognition


In [2]:
import os, json
from google.colab import userdata

os.makedirs('/root/.config/kaggle', exist_ok=True)
kaggle_creds = {
    "username": 'lukashekiladze',
    "key": 'b2f5f196e6fac2129c67a38910282180'
}
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.system('chmod 600 /root/.config/kaggle/kaggle.json')

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls

100% 285M/285M [00:02<00:00, 116MB/s]

challenges-in-representation-learning-facial-expression-recognition-challenge.zip
example_submission.csv
experiment_simple_cnn.ipynb
experiment_simple_cnn.ipynb:Zone.Identifier
fer2013.tar.gz
icml_face_data.csv
README.md
test.csv
train.csv


In [5]:
import wandb
wandb.login(key='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')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lshek22 (lshek22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

train_df = pd.read_csv('train.csv')
train_data, val_data = train_test_split(train_df, test_size=0.1, random_state=42)

class FERDataset(Dataset):
    def __init__(self, df, transform=None):
        self.labels = df['emotion'].values
        self.images = df['pixels'].apply(
            lambda x: np.array(x.split(), dtype=np.float32).reshape(48, 48)
        ).values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx] / 255.0
        img = torch.tensor(img).unsqueeze(0)
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])

train_dataset = FERDataset(train_data, transform=train_transform)
val_dataset   = FERDataset(val_data)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=64)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Device: {device}")

Train: 25838, Val: 2871, Device: cpu


In [7]:
import torch.nn as nn

class DeepCNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(DeepCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

print("Model defined")

Model defined


In [8]:
import math, torch.optim as optim

model_check = DeepCNN().to(device)
sample_imgs, sample_labels = next(iter(train_loader))
sample_imgs = sample_imgs.to(device)

with torch.no_grad():
    out = model_check(sample_imgs)

loss_val = nn.CrossEntropyLoss()(out, sample_labels.to(device))
print(f"Output shape : {out.shape}")
print(f"Initial loss : {loss_val.item():.4f} (expected ~{math.log(7):.4f})")

model_check.train()
opt = optim.Adam(model_check.parameters(), lr=0.001)
opt.zero_grad()
out = model_check(sample_imgs)
loss = nn.CrossEntropyLoss()(out, sample_labels.to(device))
loss.backward()

all_ok = all(p.grad is not None for p in model_check.parameters())
print(f"All gradients exist: {all_ok}")
print("Forward & Backward pass ok")

Output shape : torch.Size([64, 7])
Initial loss : 1.9603 (expected ~1.9459)
All gradients exist: True
Forward & Backward pass ok


In [9]:
def train_model(model, run_name, config, epochs=20):
    criterion = nn.CrossEntropyLoss()
    if config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(),
                               lr=config['lr'],
                               weight_decay=config.get('weight_decay', 0))
    else:
        optimizer = optim.SGD(model.parameters(),
                              lr=config['lr'],
                              momentum=0.9,
                              weight_decay=config.get('weight_decay', 0))

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    wandb.init(project="face expression recognition", name=run_name, config=config)

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (outputs.argmax(1) == labels).sum().item()

        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                val_correct += (outputs.argmax(1) == labels).sum().item()

        train_acc = train_correct / len(train_dataset)
        val_acc   = val_correct   / len(val_dataset)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f} | LR: {current_lr:.6f}")

        wandb.log({
            "train/loss": train_loss / len(train_loader),
            "train/accuracy": train_acc,
            "val/loss": val_loss / len(val_loader),
            "val/accuracy": val_acc,
            "learning_rate": current_lr,
            "epoch": epoch + 1
        })
        scheduler.step()

    wandb.finish()

In [10]:
model = DeepCNN(dropout_rate=0.5).to(device)
train_model(model,
    run_name="DeepCNN-Adam-lr0.001-dropout0.5",
    config={
        "architecture": "DeepCNN",
        "optimizer": "Adam",
        "lr": 0.001,
        "dropout": 0.5,
        "batchnorm": True,
        "epochs": 20,
        "note": "baseline deep cnn"
    },
    epochs=20
)

Epoch 1/20 | Train Acc: 0.249 | Val Acc: 0.315 | LR: 0.001000
Epoch 2/20 | Train Acc: 0.314 | Val Acc: 0.402 | LR: 0.001000
Epoch 3/20 | Train Acc: 0.357 | Val Acc: 0.418 | LR: 0.001000
Epoch 4/20 | Train Acc: 0.382 | Val Acc: 0.437 | LR: 0.001000
Epoch 5/20 | Train Acc: 0.398 | Val Acc: 0.458 | LR: 0.001000
Epoch 6/20 | Train Acc: 0.414 | Val Acc: 0.478 | LR: 0.001000
Epoch 7/20 | Train Acc: 0.422 | Val Acc: 0.478 | LR: 0.001000
Epoch 8/20 | Train Acc: 0.435 | Val Acc: 0.492 | LR: 0.000100
Epoch 9/20 | Train Acc: 0.442 | Val Acc: 0.504 | LR: 0.000100
Epoch 10/20 | Train Acc: 0.441 | Val Acc: 0.503 | LR: 0.000100
Epoch 11/20 | Train Acc: 0.446 | Val Acc: 0.505 | LR: 0.000100
Epoch 12/20 | Train Acc: 0.447 | Val Acc: 0.509 | LR: 0.000100
Epoch 13/20 | Train Acc: 0.446 | Val Acc: 0.506 | LR: 0.000100
Epoch 14/20 | Train Acc: 0.455 | Val Acc: 0.510 | LR: 0.000100
Epoch 15/20 | Train Acc: 0.450 | Val Acc: 0.509 | LR: 0.000010
Epoch 16/20 | Train Acc: 0.451 | Val Acc: 0.509 | LR: 0.000010
E

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
learning_rate,███████▂▂▂▂▂▂▂▁▁▁▁▁▁
train/accuracy,▁▃▅▆▆▇▇▇████████████
train/loss,█▆▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁▄▅▅▆▇▇▇████████████
val/loss,█▆▅▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
learning_rate,1e-05
train/accuracy,0.44945
train/loss,1.3996
val/accuracy,0.50958


In [11]:
model = DeepCNN(dropout_rate=0.3).to(device)
train_model(model,
    run_name="DeepCNN-Adam-lr0.001-dropout0.3",
    config={
        "architecture": "DeepCNN",
        "optimizer": "Adam",
        "lr": 0.001,
        "dropout": 0.3,
        "batchnorm": True,
        "epochs": 15,
        "note": "less dropout - expect more overfitting"
    },
    epochs=15
)

Epoch 1/15 | Train Acc: 0.284 | Val Acc: 0.367 | LR: 0.001000
Epoch 2/15 | Train Acc: 0.370 | Val Acc: 0.431 | LR: 0.001000
Epoch 3/15 | Train Acc: 0.410 | Val Acc: 0.458 | LR: 0.001000
Epoch 4/15 | Train Acc: 0.428 | Val Acc: 0.463 | LR: 0.001000
Epoch 5/15 | Train Acc: 0.450 | Val Acc: 0.490 | LR: 0.001000
Epoch 6/15 | Train Acc: 0.459 | Val Acc: 0.501 | LR: 0.001000
Epoch 7/15 | Train Acc: 0.466 | Val Acc: 0.515 | LR: 0.001000
Epoch 8/15 | Train Acc: 0.489 | Val Acc: 0.518 | LR: 0.000100
Epoch 9/15 | Train Acc: 0.491 | Val Acc: 0.531 | LR: 0.000100
Epoch 10/15 | Train Acc: 0.494 | Val Acc: 0.527 | LR: 0.000100
Epoch 11/15 | Train Acc: 0.499 | Val Acc: 0.531 | LR: 0.000100
Epoch 12/15 | Train Acc: 0.498 | Val Acc: 0.535 | LR: 0.000100
Epoch 13/15 | Train Acc: 0.499 | Val Acc: 0.535 | LR: 0.000100
Epoch 14/15 | Train Acc: 0.502 | Val Acc: 0.539 | LR: 0.000100
Epoch 15/15 | Train Acc: 0.503 | Val Acc: 0.538 | LR: 0.000010


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,███████▂▂▂▂▂▂▂▁
train/accuracy,▁▄▅▆▆▇▇████████
train/loss,█▆▄▄▃▃▂▂▁▁▁▁▁▁▁
val/accuracy,▁▄▅▅▆▆▇▇███████
val/loss,█▅▄▃▃▂▂▁▁▁▁▁▁▁▁
epoch,15
learning_rate,1e-05
train/accuracy,0.50344
train/loss,1.2893
val/accuracy,0.53814


In [12]:
model = DeepCNN(dropout_rate=0.5).to(device)
train_model(model,
    run_name="DeepCNN-Adam-lr0.001-weightdecay1e-4",
    config={
        "architecture": "DeepCNN",
        "optimizer": "Adam",
        "lr": 0.001,
        "dropout": 0.5,
        "weight_decay": 1e-4,
        "batchnorm": True,
        "epochs": 15,
        "note": "L2 regularization via weight decay"
    },
    epochs=15
)

Epoch 1/15 | Train Acc: 0.241 | Val Acc: 0.299 | LR: 0.001000
Epoch 2/15 | Train Acc: 0.302 | Val Acc: 0.371 | LR: 0.001000
Epoch 3/15 | Train Acc: 0.351 | Val Acc: 0.428 | LR: 0.001000
Epoch 4/15 | Train Acc: 0.383 | Val Acc: 0.434 | LR: 0.001000
Epoch 5/15 | Train Acc: 0.393 | Val Acc: 0.468 | LR: 0.001000
Epoch 6/15 | Train Acc: 0.413 | Val Acc: 0.469 | LR: 0.001000
Epoch 7/15 | Train Acc: 0.421 | Val Acc: 0.478 | LR: 0.001000
Epoch 8/15 | Train Acc: 0.433 | Val Acc: 0.485 | LR: 0.000100
Epoch 9/15 | Train Acc: 0.443 | Val Acc: 0.495 | LR: 0.000100
Epoch 10/15 | Train Acc: 0.446 | Val Acc: 0.499 | LR: 0.000100
Epoch 11/15 | Train Acc: 0.450 | Val Acc: 0.497 | LR: 0.000100
Epoch 12/15 | Train Acc: 0.456 | Val Acc: 0.504 | LR: 0.000100
Epoch 13/15 | Train Acc: 0.465 | Val Acc: 0.506 | LR: 0.000100
Epoch 14/15 | Train Acc: 0.462 | Val Acc: 0.503 | LR: 0.000100
Epoch 15/15 | Train Acc: 0.465 | Val Acc: 0.505 | LR: 0.000010


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,███████▂▂▂▂▂▂▂▁
train/accuracy,▁▃▄▅▆▆▇▇▇▇█████
train/loss,█▆▅▄▄▃▃▂▂▂▂▁▁▁▁
val/accuracy,▁▃▅▆▇▇▇▇███████
val/loss,█▅▄▄▃▃▃▂▂▂▂▁▁▁▁
epoch,15
learning_rate,1e-05
train/accuracy,0.46513
train/loss,1.3873
val/accuracy,0.5054
